# A vocabulary class for a tiny corpus

A language model never sees words, only whole numbers. A **vocabulary** gives each word it keeps an
id, sends every other word to one shared id for unknown words, and turns ids back into text. Here you
write one as a class, then a subclass that pads a batch of sentences to one length, the shape a model
takes as input.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import random

corpus = [
    "the model fits the data",
    "the data has noise",
    "a model with noise fits worse",
    "Data beats a bigger model",
]
for text in corpus:
    print(text)

## Part 1 (coding task): Build, encode and decode

Write a class `Vocab` with

- `__init__(self, texts, min_count=1)`: raise `ValueError` when `min_count` is below 1. Otherwise
  count every word in `texts`, after lowercasing each text with `.lower()` and cutting it into words
  with `.split()`. Keep the words that occur at least `min_count` times. Id 0 is `"<unk>"`, for
  unknown words, and the kept words get ids 1, 2, 3 and so on: the most frequent word first, with
  ties in alphabetical order. Store `self.word_to_id`, a dict from word to id that includes
  `"<unk>"`, and `self.id_to_word`, a list whose entry `i` is the word with id `i`.
- `encode(self, text)`: lowercase and split `text` the same way, and return the list of ids, with 0
  for any word the vocabulary does not hold.
- `decode(self, ids)`: return the words for `ids`, joined by single spaces. `" ".join(words)` joins
  a list of strings that way.

In [ ]:
class Vocab:
    # TODO: __init__(self, texts, min_count=1), encode(self, text) and decode(self, ids)
    def __init__(self, texts, min_count=1):
        raise NotImplementedError

    def encode(self, text):
        raise NotImplementedError

    def decode(self, ids):
        raise NotImplementedError

In [ ]:
try:
    Vocab(corpus, min_count=0)
except ValueError:
    pass
else:
    raise AssertionError("Vocab(corpus, min_count=0) should raise ValueError")

assert "Data" not in Vocab(corpus).word_to_id, "'Data' and 'data' are one word: lowercase each text before counting"
v = Vocab(corpus, min_count=2)
print(v.id_to_word)
assert v.id_to_word[0] == "<unk>", "id 0 should be '<unk>'"
assert "has" not in v.word_to_id, "'has' occurs once, so min_count=2 leaves it out"
assert v.id_to_word[1:4] == ["data", "model", "the"], f"data, model and the occur 3 times each, so ties put them in alphabetical order, got {v.id_to_word[1:4]}"
assert v.id_to_word == ["<unk>", "data", "model", "the", "a", "fits", "noise"], f"expected ['<unk>', 'data', 'model', 'the', 'a', 'fits', 'noise'], got {v.id_to_word}"
assert v.word_to_id == {w: i for i, w in enumerate(v.id_to_word)}, "word_to_id and id_to_word should agree, '<unk>' included"
rng = random.Random(0)
for _ in range(5):
    shuffled = corpus[:]
    rng.shuffle(shuffled)
    assert Vocab(shuffled, min_count=2).id_to_word == v.id_to_word, "the ids should not depend on the order of the texts"
assert v.encode("THE Data") == [3, 1], f"encode('THE Data') gave {v.encode('THE Data')}: lowercase in encode too, so 'THE' finds 'the'"
ids = v.encode("The noise beats the model")
assert ids == [3, 6, 0, 3, 2], f"encode gave {ids}, expected [3, 6, 0, 3, 2]: 'beats' is unknown, so it gets 0"
assert v.decode([1, 5, 0]) == "data fits <unk>", f"decode gave {v.decode([1, 5, 0])!r}"
assert len(Vocab(corpus).id_to_word) == 12, "with min_count=1 all 11 distinct words are kept, plus '<unk>'"
print("Part 1 passed")

## Part 2 (coding task): Pad a batch

A model takes a batch of sentences as one block of ids, so every row needs the same length. Write a
subclass `PaddedVocab(Vocab)` with

- `__init__(self, texts, min_count=1)`: run the parent's `__init__`, then add the word `"<pad>"` with
  the next free id, and store that id in `self.pad_id`. Both `word_to_id` and `id_to_word` must hold
  it.
- `encode_batch(self, texts, length)`: encode each text, cut it to its first `length` ids, and fill
  shorter ones with `pad_id` at the end, so that every row has exactly `length` ids.
- `decode(self, ids)`: like the parent's, but leave out every `pad_id`. Reuse the parent's method
  through `super()` rather than copying it.

In [ ]:
class PaddedVocab(Vocab):
    # TODO: __init__(self, texts, min_count=1), encode_batch(self, texts, length) and decode(self, ids)
    def __init__(self, texts, min_count=1):
        raise NotImplementedError

    def encode_batch(self, texts, length):
        raise NotImplementedError

    def decode(self, ids):
        raise NotImplementedError

In [ ]:
try:
    PaddedVocab(corpus, min_count=0)
except ValueError:
    pass
else:
    raise AssertionError("PaddedVocab(corpus, min_count=0) should raise ValueError: run the parent's __init__")

pv = PaddedVocab(corpus, min_count=2)
assert isinstance(pv, Vocab), "PaddedVocab should inherit from Vocab"
assert pv.pad_id == 7, f"pad_id should be the next free id, 7, got {pv.pad_id}"
assert pv.pad_id != 0, "id 0 already means '<unk>', so padding needs its own id"
assert len(pv.id_to_word) == 8 and pv.id_to_word[7] == "<pad>", "id_to_word should end with '<pad>'"
assert pv.word_to_id.get("<pad>") == 7, "word_to_id should map '<pad>' to its id"
batch = pv.encode_batch(["the data", "a model fits the noise well"], 4)
print(batch)
assert [len(row) for row in batch] == [4, 4], f"every row should have 4 ids, got lengths {[len(row) for row in batch]}"
assert batch == [[3, 1, 7, 7], [4, 2, 5, 3]], f"expected [[3, 1, 7, 7], [4, 2, 5, 3]], got {batch}"
assert pv.decode([3, 1, 7, 7]) == "the data", f"decode should drop the padding, got {pv.decode([3, 1, 7, 7])!r}"
assert pv.decode([0, 2]) == "<unk> model", "decode should still show '<unk>'"
print("Part 2 passed")

## Solution

### Part 1: Build, encode and decode

In [ ]:
class Vocab:
    def __init__(self, texts, min_count=1):
        if min_count < 1:
            raise ValueError("min_count must be at least 1")
        counts = {}
        for text in texts:
            for word in text.lower().split():
                counts[word] = counts.get(word, 0) + 1
        kept = [w for w in counts if counts[w] >= min_count]
        kept = sorted(kept, key=lambda w: (-counts[w], w))
        self.id_to_word = ["<unk>"] + kept
        self.word_to_id = {w: i for i, w in enumerate(self.id_to_word)}

    def encode(self, text):
        return [self.word_to_id.get(w, 0) for w in text.lower().split()]

    def decode(self, ids):
        return " ".join(self.id_to_word[i] for i in ids)

**Why it works.** A dict with `counts.get(word, 0) + 1` counts the words in one pass. The key `(-count, word)` puts the
most frequent words first and settles ties alphabetically, so `data`, `model` and `the`, three times
each, get ids 1, 2 and 3. Without it they would keep the order of their first appearance. Building
`word_to_id` from `id_to_word` with `enumerate` keeps the two in step, and `.get(w, 0)` sends every
unknown word to `<unk>`. Lowercasing before counting matters: `Data` in the last sentence is the
third `data`, and without it `data` would fall behind `model` and `the`.

### Part 2: Pad a batch

In [ ]:
class PaddedVocab(Vocab):
    def __init__(self, texts, min_count=1):
        super().__init__(texts, min_count)
        self.pad_id = len(self.id_to_word)
        self.id_to_word.append("<pad>")
        self.word_to_id["<pad>"] = self.pad_id

    def encode_batch(self, texts, length):
        batch = []
        for text in texts:
            ids = self.encode(text)[:length]
            batch.append(ids + [self.pad_id] * (length - len(ids)))
        return batch

    def decode(self, ids):
        return super().decode([i for i in ids if i != self.pad_id])

**Why it works.** `super().__init__` builds the counts, ids and both lookups, so the subclass only appends `"<pad>"` at
the end, where its id is `len(self.id_to_word)` before the append, 7 here. Cutting with `[:length]`
and adding `[self.pad_id] * (length - len(ids))` handles both cases at once: a long text loses its
tail, and for a short one the repeated list fills the gap, since a list times 0 is empty. The new
`decode` filters the ids and hands the rest to the parent's `decode`, so the joining logic lives in
one place.